# Evaluasi Lengkap, Grad-CAM & Analisis Error — Klasifikasi Topeng Bali
## Target Pertemuan 18 November 2026 (versi otomatis)

Semua metrik, confusion matrix, daftar error, dan peta Grad-CAM **dihitung langsung** dari file hasil training. Tidak ada angka yang diketik manual, dan **tidak ada path yang harus diisi manual**: notebook mencari sendiri file yang dibutuhkan di Google Drive / `/content`.

### Yang harus disiapkan SEBELUM menjalankan (cukup letakkan di Google Drive)

```
MyDrive/
└── TopengBali/                          <- nama folder bebas, boleh di mana saja di MyDrive
    ├── repo/                            <- folder proyek (atau upload sebagai repo.zip)
    │   └── api/
    │       ├── __init__.py
    │       └── model.py                 <- harus berisi VARIANTS dan build_variant
    ├── outputs_mobilenetv3/             <- hasil MobileNetV3_Attention.ipynb
    │   ├── split_leakage_free.csv
    │   └── <varian>/seed_42/
    │       ├── best_model.pth
    │       └── metrics.json
    ├── BaliMask_Clean/train/<kelas>/*.jpg    <- gambar dataset (atau dataset .zip)
    └── real_world/<kelas>/*.jpg         <- OPSIONAL: foto lapangan sendiri
```

Lalu: **Runtime > Run all**. Sel 2 mencetak daftar centang (OK / GAGAL) beserta lokasi yang ditemukan. Jika ada yang GAGAL, pesan errornya menjelaskan apa yang harus diletakkan.

**Catatan perbandingan yang adil:** semua model dievaluasi pada **test set yang sama** (`split == "test"`). Model lain (VGG16, EfficientNetV2-S, ConvNeXt) hanya boleh dimasukkan bila dilatih ulang dengan CSV split yang sama.

In [ ]:
# Sel 1 - Setup: install paket, mount Drive, konfigurasi
%pip -q install grad-cam seaborn scikit-learn

import os, sys, json, copy, time, zipfile, shutil
from pathlib import Path
from collections import Counter

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# Biarkan None = otomatis. Isi HANYA bila auto-deteksi memilih lokasi yang salah.
OVERRIDE = {
    "OUT_ROOT": None,    # folder yang berisi split_leakage_free.csv, contoh: "/content/drive/MyDrive/TopengBali/outputs_mobilenetv3"
    "DATA_DIR": None,    # folder induk kelas gambar, contoh: "/content/drive/MyDrive/TopengBali/BaliMask_Clean/train"
    "API_PARENT": None,  # folder yang berisi folder api/, contoh: "/content/drive/MyDrive/TopengBali/repo"
}
SEED = 42        # otomatis diganti bila hanya ada seed lain di checkpoint
IMG_SIZE = 224
print("Colab:", IN_COLAB)

In [ ]:
# Sel 2 - Auto-deteksi: split CSV, kode api/, dataset gambar + daftar centang kesiapan
import pandas as pd

t0 = time.time()
SEARCH_ROOTS = []
for r in [Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives"), Path("/content"), Path.cwd()]:
    if r.exists() and r not in SEARCH_ROOTS:
        SEARCH_ROOTS.append(r)

SKIP = {"__pycache__", "node_modules", "site-packages", "dist-packages", "sample_data", "repo_extracted", "dataset_extracted"}

def _walk(root, max_depth=7):
    root = Path(root); base = len(root.parts)
    for dp, dn, fn in os.walk(root):
        dpp = Path(dp)
        dn[:] = [d for d in dn
                 if d not in SKIP and not d.startswith(".")
                 and not ((dpp / d) in SEARCH_ROOTS and (dpp / d) != root)]
        if len(dpp.parts) - base >= max_depth:
            dn[:] = []
        yield dpp, dn, fn

def _newest(paths):
    paths = list(dict.fromkeys(paths))
    return sorted(paths, key=lambda p: p.stat().st_mtime if p.exists() else 0, reverse=True)

found = {"csv": [], "api": [], "zips": [], "real_world": []}
for r in SEARCH_ROOTS:
    for dp, dn, fn in _walk(r):
        if "split_leakage_free.csv" in fn: found["csv"].append(dp / "split_leakage_free.csv")
        if dp.name == "api" and "model.py" in fn: found["api"].append(dp.parent)
        if dp.name == "real_world": found["real_world"].append(dp)
        found["zips"] += [dp / f for f in fn if f.lower().endswith(".zip")]
print(f"Pemindaian selesai ({time.time()-t0:.0f} dtk)")

# ---------- 1) split_leakage_free.csv -> OUT_ROOT ----------
HELP_CSV = ("split_leakage_free.csv tidak ditemukan. Jalankan dulu MobileNetV3_Attention.ipynb (Sel 3 dan Sel 8), "
            "lalu pastikan folder 'outputs_mobilenetv3' ada di Google Drive (MyDrive/...).")
if OVERRIDE["OUT_ROOT"]:
    OUT_ROOT = Path(OVERRIDE["OUT_ROOT"])
elif os.environ.get("OUT_ROOT") and (Path(os.environ["OUT_ROOT"]) / "split_leakage_free.csv").exists():
    OUT_ROOT = Path(os.environ["OUT_ROOT"])
else:
    csvs = _newest(found["csv"])
    if not csvs: raise FileNotFoundError(HELP_CSV)
    n_ckpt = lambda c: len(list(c.parent.glob("*/seed_*/best_model.pth")))
    csvs.sort(key=lambda c: (n_ckpt(c), c.stat().st_mtime), reverse=True)   # utamakan folder yang punya checkpoint terbanyak
    OUT_ROOT = csvs[0].parent
    if len(csvs) > 1: print("Peringatan: ada >1 split CSV, dipakai:", csvs[0], "\n  lainnya:", [str(c) for c in csvs[1:]])
SPLIT_CSV = OUT_ROOT / "split_leakage_free.csv"
assert SPLIT_CSV.exists(), HELP_CSV
EVAL_DIR = OUT_ROOT / "evaluasi"; EVAL_DIR.mkdir(exist_ok=True)

# ---------- 2) folder api/ (berisi model.py) ----------
def _api_from_zip():
    for z in _newest(found["zips"]):
        try:
            with zipfile.ZipFile(z) as zf:
                hit = [n for n in zf.namelist() if n.endswith("api/model.py") and "__MACOSX" not in n]
                if not hit: continue
                out = Path("/content/repo_extracted") if Path("/content").exists() else Path("repo_extracted")
                zf.extractall(out)
                return out / hit[0][:-len("api/model.py")]
        except Exception:
            continue
    return None

HELP_API = ("Folder 'api/' (berisi model.py dengan VARIANTS & build_variant) tidak ditemukan. Letakkan folder proyek di Google Drive "
            "(contoh MyDrive/TopengBali/repo/api/model.py), atau upload sebagai repo.zip, lalu jalankan ulang Sel 2.")
if OVERRIDE["API_PARENT"]: API_PARENT = Path(OVERRIDE["API_PARENT"])
elif found["api"]:         API_PARENT = _newest([p / "api" / "model.py" for p in found["api"]])[0].parent.parent
else:                      API_PARENT = _api_from_zip()
if API_PARENT is None or not (API_PARENT / "api" / "model.py").exists(): raise FileNotFoundError(HELP_API)
sys.path.insert(0, str(API_PARENT.resolve()))

# ---------- 3) folder gambar dataset -> DATA_DIR ----------
df = pd.read_csv(SPLIT_CSV)
_p0 = Path(str(df.path.iloc[0]))
N_PARTS = len(_p0.parts) if not _p0.is_absolute() else 2        # biasanya "<kelas>/<file>.jpg"
def _tail(p): return Path(*Path(str(p)).parts[-N_PARTS:])
_t0 = _tail(df.path.iloc[0])

def _find_data_root():
    cands = []
    if OVERRIDE["DATA_DIR"]: cands.append(Path(OVERRIDE["DATA_DIR"]))
    if os.environ.get("DATA_DIR"): cands.append(Path(os.environ["DATA_DIR"]))
    cands += [OUT_ROOT, OUT_ROOT.parent, Path("/content/BaliMask_Clean/train"), Path("/content/BaliMask_Clean")]
    for c in cands:
        if (c / _t0).exists(): return c
    for r in SEARCH_ROOTS:                                         # telusuri berdasarkan nama file contoh
        for dp, dn, fn in _walk(r, 9):
            if _t0.name in fn:
                root = dp
                for _ in range(N_PARTS - 1): root = root.parent
                if (root / _t0).exists(): return root
    for z in _newest(found["zips"]):                               # terakhir: ekstrak dari zip dataset
        try:
            with zipfile.ZipFile(z) as zf:
                hit = [n for n in zf.namelist() if n.endswith(_t0.as_posix())]
                if not hit: continue
                out = Path("/content/dataset_extracted") if Path("/content").exists() else Path("dataset_extracted")
                print("Mengekstrak dataset dari", z.name, "...")
                zf.extractall(out)
                root = out / hit[0][:-len(_t0.as_posix())]
                if (root / _t0).exists(): return root
        except Exception:
            continue
    return None

DATA_DIR = _find_data_root()
HELP_DATA = (f"Gambar dataset tidak ditemukan (dicari: .../{_t0.as_posix()}). Letakkan folder dataset yang sama dengan saat training "
             "(mis. MyDrive/TopengBali/BaliMask_Clean/train/<kelas>/*.jpg) atau dataset .zip di Google Drive.")
if DATA_DIR is None: raise FileNotFoundError(HELP_DATA)
def resolve_img(p): return DATA_DIR / _tail(p)

# ---------- 4) real_world (opsional) ----------
REAL_DIR = None
if found["real_world"]:
    share = lambda p: len(os.path.commonpath([str(p), str(DATA_DIR)]))
    REAL_DIR = sorted(found["real_world"], key=share, reverse=True)[0]

# ---------- Daftar centang ----------
miss = [p for p in df[df.split == "test"].path if not resolve_img(p).exists()]
miss_all = [p for p in df.path if not resolve_img(p).exists()]
n_test = int((df.split == "test").sum())
ok = lambda b: "OK    " if b else "GAGAL "
print("\n" + "=" * 78 + "\nDAFTAR KESIAPAN\n" + "=" * 78)
print(ok(True),  "split CSV      :", SPLIT_CSV)
print(ok(True),  "kode api/      :", API_PARENT / "api")
print(ok(not miss), f"gambar dataset : {DATA_DIR}  (test ditemukan {n_test-len(miss)}/{n_test}; seluruh CSV {len(df)-len(miss_all)}/{len(df)})")
print(ok(True) if REAL_DIR else "SKIP  ", "real_world (opsional):", REAL_DIR or "tidak ada -> Sel 10 hanya memberi peringatan")
print("       output evaluasi:", EVAL_DIR)
assert not miss, f"{len(miss)} gambar test tidak ditemukan di {DATA_DIR}, contoh: {miss[:3]}. {HELP_DATA}"

In [ ]:
# Sel 3 - Import & muat split
import numpy as np, torch
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
from torchvision import transforms
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report, precision_recall_fscore_support)
from IPython.display import display
from api.model import VARIANTS, build_variant

sns.set_theme(style="whitegrid")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "" if device.type == "cuda" else "(CPU: tetap jalan, hanya lebih lambat. Colab: Runtime > Change runtime type > GPU)")

CLASSES = df.drop_duplicates("y").sort_values("y").label.tolist()
NUM_CLASSES = len(CLASSES)
te_df = df[df.split == "test"].reset_index(drop=True)

# Verifikasi integritas: satu grup (objek topeng fisik) hanya boleh ada di satu split
assert df.groupby("group").split.nunique().max() == 1, "Ada grup yang bocor antar-split!"
print(f"{NUM_CLASSES} kelas | train/val/test = "
      f"{(df.split=='train').sum()}/{(df.split=='val').sum()}/{len(te_df)} | split bebas bocor: OK")
display(pd.crosstab(df.label, df.split)[["train", "val", "test"]])

In [ ]:
# Sel 4 - Muat semua checkpoint yang tersedia (seed terdeteksi otomatis)
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
eval_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)), transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
view_tf = transforms.Compose([transforms.Resize(int(IMG_SIZE * 1.14)), transforms.CenterCrop(IMG_SIZE)])

# Auto-pilih seed: pakai SEED bila ada, kalau tidak pakai seed yang paling banyak punya checkpoint
_seeds = Counter(p.parent.name for p in OUT_ROOT.glob("*/seed_*/best_model.pth"))
if _seeds and f"seed_{SEED}" not in _seeds:
    SEED = int(_seeds.most_common(1)[0][0].split("_")[1])
    print(f"Checkpoint seed_42 tidak ada; otomatis memakai seed_{SEED}")
print("Checkpoint ditemukan per seed:", dict(_seeds) or "tidak ada")

def _clean_state(sd):
    if isinstance(sd, dict):
        for k in ("state_dict", "model_state_dict", "model"):
            if k in sd and isinstance(sd[k], dict): sd = sd[k]; break
    return {k.replace("module.", "", 1): v for k, v in sd.items()}

models_loaded, skipped = {}, {}
for tag in VARIANTS:
    ckpt = OUT_ROOT / tag / f"seed_{SEED}" / "best_model.pth"
    if not ckpt.exists():
        skipped[tag] = "belum dilatih (checkpoint tidak ada)"; continue
    try:
        m = build_variant(tag, NUM_CLASSES, pretrained=False)
        m.load_state_dict(_clean_state(torch.load(ckpt, map_location="cpu")))
        models_loaded[tag] = m.to(device).eval()
    except Exception as e:
        skipped[tag] = f"gagal dimuat: {type(e).__name__}: {str(e)[:150]}"

extra = sorted({p.parent.parent.name for p in OUT_ROOT.glob("*/seed_*/best_model.pth")} - set(VARIANTS))
assert models_loaded, "Belum ada checkpoint yang bisa dimuat. Latih minimal satu varian di MobileNetV3_Attention.ipynb (Sel 8)."
print("Dievaluasi :", list(models_loaded))
for t, why in skipped.items(): print(f"Dilewati   : {t} -> {why}")
if extra: print("Catatan    : folder checkpoint di luar VARIANTS (tidak dievaluasi):", extra)

In [ ]:
# Sel 5 - Prediksi test set + metrik (semua dihitung, tidak ada angka manual)
@torch.no_grad()
def predict_df(model, frame):
    probs = []
    for i in range(0, len(frame), 32):
        batch = torch.stack([eval_tf(Image.open(resolve_img(p)).convert("RGB")) for p in frame.path[i:i+32]])
        probs.append(torch.softmax(model(batch.to(device)), 1).cpu().numpy())
    return np.concatenate(probs)

def wilson(k, n, z=1.96):
    if n == 0: return (0, 0)
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z * np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return max(0, c-h), min(1, c+h)

y_true = te_df.y.values
results, preds = [], {}
for tag, m in models_loaded.items():
    probs = predict_df(m, te_df)
    pred = probs.argmax(1)
    preds[tag] = dict(probs=probs, pred=pred)
    k = int((pred == y_true).sum())
    lo, hi = wilson(k, len(y_true))
    meta = {}
    mj = OUT_ROOT / tag / f"seed_{SEED}" / "metrics.json"
    if mj.exists():
        with open(mj) as f: meta = json.load(f)
    results.append({
        "Model": tag, "n_test": len(y_true), "Salah": len(y_true) - k,
        "Akurasi (%)": round(100 * k / len(y_true), 2),
        "CI95 Wilson (%)": f"{100*lo:.1f}-{100*hi:.1f}",
        "Macro-F1 (%)": round(100 * f1_score(y_true, pred, average="macro", zero_division=0), 2),
        "Weighted-F1 (%)": round(100 * f1_score(y_true, pred, average="weighted", zero_division=0), 2),
        "Param (M)": meta.get("params_m"), "Ukuran (MB)": meta.get("size_mb"),
        "FLOPs (G)": meta.get("gflops"), "Latensi CPU (ms)": meta.get("latency_cpu_ms"),
    })
    out = te_df[["path", "label", "y"]].copy()
    out["pred"] = pred; out["pred_label"] = [CLASSES[i] for i in pred]
    out["confidence"] = probs.max(1)
    out.to_csv(OUT_ROOT / tag / f"seed_{SEED}" / "test_predictions.csv", index=False)

table = pd.DataFrame(results).set_index("Model")
display(table)
table.to_csv(EVAL_DIR / "evaluation_summary.csv")

### Cara membaca tabel
Dengan test set sekecil ini dan akurasi mendekati 100%, selisih 1-2 gambar antar varian **belum tentu bermakna**. Kolom `CI95 Wilson` menunjukkan rentang ketidakpastian akurasi; jika rentang antar varian saling tumpang tindih, jangan menyimpulkan satu varian lebih baik hanya dari angka akurasi. Gunakan `SEEDS = [42, 123, 2024]` di notebook training bila ingin klaim ablation yang lebih kuat.

In [ ]:
# Sel 6 - Laporan per kelas + Confusion Matrix
for tag, r in preds.items():
    print("=" * 70, f"\n{tag}\n", "=" * 70)
    print(classification_report(y_true, r["pred"], labels=list(range(NUM_CLASSES)),
                                target_names=CLASSES, digits=3, zero_division=0))

n = len(preds)
fig, axes = plt.subplots(1, n, figsize=(7.5 * n, 6.5), squeeze=False)
for ax, (tag, r) in zip(axes[0], preds.items()):
    cm = confusion_matrix(y_true, r["pred"], labels=list(range(NUM_CLASSES)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_title(f"{tag}  (salah: {int((r['pred']!=y_true).sum())})", fontweight="bold")
    ax.set_xlabel("Prediksi"); ax.set_ylabel("Sebenarnya"); ax.tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.savefig(EVAL_DIR / "confusion_matrices.png", dpi=120); plt.show()

In [ ]:
# Sel 7 - Analisis error: pasangan kelas yang tertukar + daftar gambar salah
rows = []
for tag, r in preds.items():
    for i in np.where(r["pred"] != y_true)[0]:
        rows.append({"Model": tag, "path": te_df.path[i], "Sebenarnya": CLASSES[y_true[i]],
                     "Prediksi": CLASSES[r["pred"][i]], "Confidence": round(float(r["probs"][i].max()), 3),
                     "p(kelas benar)": round(float(r["probs"][i][y_true[i]]), 3)})
err = pd.DataFrame(rows)
if err.empty:
    print("Tidak ada kesalahan prediksi pada test set untuk model yang dievaluasi.")
else:
    pairs = (err.groupby(["Model", "Sebenarnya", "Prediksi"]).size()
                .rename("Jumlah").reset_index().sort_values("Jumlah", ascending=False))
    print("Pasangan kelas yang paling sering tertukar:"); display(pairs.head(15))
    print("Gambar yang salah diprediksi:"); display(err.sort_values("Confidence", ascending=False).head(30))
    err.to_csv(EVAL_DIR / "error_analysis.csv", index=False)

    show = err.drop_duplicates("path").head(12)
    cols = 4; rws = int(np.ceil(len(show) / cols))
    fig, axes = plt.subplots(rws, cols, figsize=(4 * cols, 4.2 * rws), squeeze=False)
    for ax in axes.ravel(): ax.axis("off")
    for ax, (_, e) in zip(axes.ravel(), show.iterrows()):
        ax.imshow(view_tf(Image.open(resolve_img(e["path"])).convert("RGB")))
        ax.set_title(f"asli: {e['Sebenarnya']}\nprediksi: {e['Prediksi']} ({e['Confidence']:.2f})", fontsize=9)
    plt.tight_layout(); plt.savefig(EVAL_DIR / "error_gallery.png", dpi=110); plt.show()

### Template interpretasi error (isi setelah melihat hasil Sel 7)
Jangan menulis penjelasan penyebab error sebelum melihat gambarnya. Untuk tiap pasangan tertukar teratas, cek langsung gambar di galeri: apakah penyebabnya pencahayaan/pantulan, sudut foto, kemiripan morfologi, atau label dataset yang memang ambigu? Tulis hanya yang terlihat pada gambar dan didukung peta Grad-CAM di bawah.

In [ ]:
# Sel 8 - Grad-CAM: 1 gambar uji benar per kelas, dibandingkan antar varian
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

_cam_layer = {}
def get_cam_layer(tag, model):
    # Layer target dideteksi otomatis: modul terakhir di model.features yang masih berupa peta fitur 4D
    # (pada MobileNetV3 standar = blok konv 960-channel terakhir). Bekerja juga untuk varian attention.
    if tag in _cam_layer: return _cam_layer[tag]
    feats = getattr(model, "features", None)
    mods = list(feats.children()) if feats is not None else [m for m in model.modules() if isinstance(m, torch.nn.Conv2d)]
    shapes = {}
    hooks = [m.register_forward_hook(lambda mod, i, o, k=k: shapes.__setitem__(k, tuple(o.shape)) if hasattr(o, "shape") else None)
             for k, m in enumerate(mods)]
    with torch.no_grad(): model(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=device))
    for h in hooks: h.remove()
    idx = max(k for k, s in shapes.items() if len(s) == 4)
    _cam_layer[tag] = mods[idx]
    print(f"[Grad-CAM] {tag}: layer target = features[{idx}] ({type(mods[idx]).__name__}), keluaran {shapes[idx]}")
    return mods[idx]

def cam_for(tag, model, x, cls):
    cam = GradCAM(model=model, target_layers=[get_cam_layer(tag, model)])
    try:
        return cam(input_tensor=x, targets=[ClassifierOutputTarget(int(cls))])[0]
    finally:
        try: cam.activations_and_grads.release()
        except Exception: pass

rng = np.random.RandomState(SEED)
first_tag = next(iter(preds))
pick = []
for c in range(NUM_CLASSES):
    idx = np.where((y_true == c) & (preds[first_tag]["pred"] == c))[0]
    if len(idx): pick.append(int(rng.choice(idx)))

tags = list(models_loaded)
fig, axes = plt.subplots(len(pick), len(tags) + 1, figsize=(3.2 * (len(tags) + 1), 3.2 * len(pick)), squeeze=False)
for r_i, i in enumerate(pick):
    img = Image.open(resolve_img(te_df.path[i])).convert("RGB")
    rgb = np.asarray(view_tf(img)).astype(np.float32) / 255.0
    x = eval_tf(img).unsqueeze(0).to(device)
    axes[r_i, 0].imshow(rgb); axes[r_i, 0].set_title(CLASSES[y_true[i]], fontsize=9, fontweight="bold")
    for c_i, tag in enumerate(tags, start=1):
        g = cam_for(tag, models_loaded[tag], x.clone().requires_grad_(True), y_true[i])
        axes[r_i, c_i].imshow(show_cam_on_image(rgb, g, use_rgb=True))
        axes[r_i, c_i].set_title(tag.replace("mobilenetv3_", ""), fontsize=9)
for ax in axes.ravel(): ax.axis("off")
plt.tight_layout(); plt.savefig(EVAL_DIR / "gradcam_per_class.png", dpi=110); plt.show()

In [ ]:
# Sel 9 - Grad-CAM pada gambar yang SALAH diprediksi (kelas benar vs kelas yang diprediksi)
if err.empty:
    print("Tidak ada error untuk divisualisasikan.")
else:
    sel = err.drop_duplicates(["Model", "path"]).head(6)
    fig, axes = plt.subplots(len(sel), 3, figsize=(10, 3.4 * len(sel)), squeeze=False)
    for r_i, (_, e) in enumerate(sel.iterrows()):
        m = models_loaded[e["Model"]]
        img = Image.open(resolve_img(e["path"])).convert("RGB")
        rgb = np.asarray(view_tf(img)).astype(np.float32) / 255.0
        x = eval_tf(img).unsqueeze(0).to(device)
        t, p = CLASSES.index(e["Sebenarnya"]), CLASSES.index(e["Prediksi"])
        axes[r_i, 0].imshow(rgb); axes[r_i, 0].set_title(f"{e['Model'].replace('mobilenetv3_','')}\nasli: {e['Sebenarnya']}", fontsize=9)
        axes[r_i, 1].imshow(show_cam_on_image(rgb, cam_for(e["Model"], m, x.clone().requires_grad_(True), t), use_rgb=True))
        axes[r_i, 1].set_title(f"CAM kelas benar: {e['Sebenarnya']}", fontsize=9)
        axes[r_i, 2].imshow(show_cam_on_image(rgb, cam_for(e["Model"], m, x.clone().requires_grad_(True), p), use_rgb=True))
        axes[r_i, 2].set_title(f"CAM kelas prediksi: {e['Prediksi']}", fontsize=9)
    for ax in axes.ravel(): ax.axis("off")
    plt.tight_layout(); plt.savefig(EVAL_DIR / "gradcam_errors.png", dpi=110); plt.show()

In [ ]:
# Sel 10 - (Opsional) Uji foto lapangan asli. Folder real_world/<kelas>/*.jpg terdeteksi otomatis (lihat Sel 2).
if REAL_DIR is None:
    print("Tidak ada folder real_world -> klaim akurasi di dunia nyata BELUM bisa diberikan. "
          "Buat folder real_world/<nama_kelas>/*.jpg di Google Drive (nama kelas harus sama dengan CLASSES), minimal beberapa foto per kelas.")
else:
    unknown = [d.name for d in REAL_DIR.iterdir() if d.is_dir() and d.name not in CLASSES]
    if unknown: print("Peringatan: folder kelas tidak dikenali (nama harus sama dengan CLASSES), dilewati:", unknown)
    rows = []
    for c in CLASSES:
        for p in sorted((REAL_DIR / c).glob("*")) if (REAL_DIR / c).exists() else []:
            if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}:
                rows.append((str(p), CLASSES.index(c)))
    if not rows:
        print("Folder real_world ada tetapi kosong:", REAL_DIR)
    else:
        real = pd.DataFrame(rows, columns=["path", "y"])
        print(f"real_world: {REAL_DIR}  (n={len(real)})")
        for tag, m in models_loaded.items():
            with torch.no_grad():
                pr = [m(eval_tf(Image.open(p).convert("RGB")).unsqueeze(0).to(device)).argmax(1).item() for p in real.path]
            print(f"{tag:28s} akurasi foto lapangan: {100*np.mean(np.array(pr)==real.y.values):.2f}%")

In [ ]:
# Sel 11 - Ringkasan Markdown untuk README/laporan (dibuat dari hasil di atas)
fmt = lambda v: "-" if v is None or (isinstance(v, float) and np.isnan(v)) else v
lines = ["| Model | Akurasi (%) | Macro-F1 (%) | Salah/n | Param (M) | Ukuran (MB) | Latensi CPU (ms) |", "|---|---|---|---|---|---|---|"]
for tag, r in table.iterrows():
    lines.append(f"| {tag} | {r['Akurasi (%)']} | {r['Macro-F1 (%)']} | {r['Salah']}/{r['n_test']} | "
                 f"{fmt(r['Param (M)'])} | {fmt(r['Ukuran (MB)'])} | {fmt(r['Latensi CPU (ms)'])} |")
md_out = "\n".join(lines)
(EVAL_DIR / "results_summary.md").write_text(md_out, encoding="utf-8")
print(md_out)

In [ ]:
# Sel 12 - Simpan hasil: salin ke Google Drive (jika belum di Drive) + zip untuk diunduh
drive_root = Path("/content/drive/MyDrive")
if IN_COLAB and drive_root.exists() and not str(EVAL_DIR.resolve()).startswith(str(drive_root)):
    dst = drive_root / "TopengBali_Evaluasi"
    shutil.copytree(EVAL_DIR, dst, dirs_exist_ok=True)
    print("Hasil disalin ke Drive:", dst)
else:
    print("Hasil tersimpan di:", EVAL_DIR)
zip_base = Path("/content/hasil_evaluasi") if Path("/content").exists() else Path("hasil_evaluasi")
zip_path = shutil.make_archive(str(zip_base), "zip", EVAL_DIR)
print("Zip:", zip_path)
for f in sorted(EVAL_DIR.iterdir()): print(" -", f.name)
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    pass